In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 183kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.27MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 6.25MB/s]


In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using mps device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


In [8]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301688  [   64/60000]
loss: 2.291489  [ 6464/60000]
loss: 2.262091  [12864/60000]
loss: 2.249424  [19264/60000]
loss: 2.256127  [25664/60000]
loss: 2.213927  [32064/60000]
loss: 2.222345  [38464/60000]
loss: 2.188783  [44864/60000]
loss: 2.178621  [51264/60000]
loss: 2.143984  [57664/60000]
Test Error: 
 Accuracy: 40.8%, Avg loss: 2.141784 

Epoch 2
-------------------------------
loss: 2.160577  [   64/60000]
loss: 2.153222  [ 6464/60000]
loss: 2.079833  [12864/60000]
loss: 2.085776  [19264/60000]
loss: 2.057284  [25664/60000]
loss: 1.988818  [32064/60000]
loss: 2.015978  [38464/60000]
loss: 1.933761  [44864/60000]
loss: 1.936858  [51264/60000]
loss: 1.863097  [57664/60000]
Test Error: 
 Accuracy: 55.2%, Avg loss: 1.859168 

Epoch 3
-------------------------------
loss: 1.903227  [   64/60000]
loss: 1.875379  [ 6464/60000]
loss: 1.740934  [12864/60000]
loss: 1.774017  [19264/60000]
loss: 1.683313  [25664/60000]
loss: 1.636312  [32064/600

In [11]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [12]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [13]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


In [22]:
"""
linear_regression_pytorch.py
============================
The simplest possible end-to-end PyTorch example: fit a straight line.

Goal: recover the line  y = 2*x + 1  from noisy data, using gradient descent.
We KNOW the answer (weight=2, bias=1), so we can check the model learns it.

Run it:      python linear_regression_pytorch.py
Requires:    pip install torch

This file is annotated to map every line to the concepts you've learned:
forward pass, loss, backward (backprop), optimizer step, epochs.
"""

import torch
import torch.nn as nn

torch.manual_seed(0)  # reproducible results

# ----------------------------------------------------------------------
# 1. MAKE SOME DATA
# ----------------------------------------------------------------------
# 100 x-values, shape (100, 1). The extra "1" is the feature dimension:
# each example is a vector of length 1. Models expect (n_examples, n_features).
X = torch.linspace(-3, 3, 100).reshape(-1, 1)

TRUE_W, TRUE_B = 2.0, 1.0
noise = 0.5 * torch.randn(X.shape)          # random noise so it's not a perfect line
y = TRUE_W * X + TRUE_B + noise             # the targets we want to predict

# ----------------------------------------------------------------------
# 2. DEFINE THE MODEL
# ----------------------------------------------------------------------
# nn.Linear(1, 1) is literally "y = w*x + b" with ONE learnable weight w
# and ONE learnable bias b. Both start at random values; training fixes them.
#   in_features=1  -> each input is 1 number
#   out_features=1 -> each output is 1 number

class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        return self.linear(x)

model = LinearRegression()
print(model)


print("BEFORE training (random init):")
# print(f"  weight = {model.weight.item():.3f}   bias = {model.bias.item():.3f}\n")

# ----------------------------------------------------------------------
# 3. PICK A LOSS FUNCTION AND AN OPTIMIZER
# ----------------------------------------------------------------------
# Loss: Mean Squared Error = average of (prediction - target)^2.
# It's a single number measuring "how wrong are we right now?"
loss_fn = nn.MSELoss()

# Optimizer: the thing that adjusts the weights. SGD = Stochastic Gradient
# Descent. lr (learning rate) = how big a step to take each update.
# We hand it model.parameters() so it knows WHICH numbers it's allowed to tweak.
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# ----------------------------------------------------------------------
# 4. THE TRAINING LOOP  (the 5 lines that show up in every PyTorch project)
# ----------------------------------------------------------------------
num_epochs = 100   # how many times we pass over the whole dataset
for epoch in range(num_epochs):

    # --- forward pass: run the model to get predictions ---
    y_pred = model(X)                 # calls model.forward(X) under the hood

    # --- compute the loss: how wrong were those predictions? ---
    loss = loss_fn(y_pred, y)

    # --- backward pass: backprop computes the gradient of the loss
    #     w.r.t. every weight (how to change each one to reduce loss) ---
    optimizer.zero_grad()             # clear gradients left over from last step
    loss.backward()                   # <- this is backpropagation

    # --- update: nudge every weight a little bit downhill ---
    optimizer.step()

    # print progress every 20 epochs
    # if (epoch + 1) % 20 == 0:
    #     w = model.weight.item()
    #     b = model.bias.item()
    #     print(f"epoch {epoch+1:3d} | loss {loss.item():.4f} | "
    #           f"weight {w:.3f} | bias {b:.3f}")

# ----------------------------------------------------------------------
# 5. CHECK WHAT IT LEARNED
# ----------------------------------------------------------------------
print("\nAFTER training:")
print(f"  learned weight = {model.linear.weight.item():.3f}  (true value: {TRUE_W})")
print(f"  learned bias   = {model.linear.bias.item():.3f}  (true value: {TRUE_B})")
print("\nThe model started from random numbers and, through ~100 small")
print("gradient-descent steps, recovered the line we hid in the data.")


LinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
BEFORE training (random init):

AFTER training:
  learned weight = 2.000  (true value: 2.0)
  learned bias   = 1.019  (true value: 1.0)

The model started from random numbers and, through ~100 small
gradient-descent steps, recovered the line we hid in the data.
